In [10]:
import numpy as np
import scipy.io as sio
import scipy.stats as stats
import scipy.signal as signal
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

FS = 12_000
WINDOW_SIZE = FS
STEP_SIZE = FS // 2

In [23]:
def load_class_signals(paths: list) -> np.ndarray:
    """Load multiple .mat files and concatenate into one signal"""
    segments=[]
    for path in paths:
        mat = sio.loadmat(Path(path), squeeze_me=True)
        de_keys = [k for k in mat.keys() if "DE" in k and "time" in k]
        if not de_keys:
            raise KeyError(f"No DE key found in {path}")
        segments.append(mat[de_keys[0]].astype(np.float64))
    return np.concatenate(segments)

signals = {
    "Normal": load_class_signals(["../data/normal/97.mat", 
                                  "../data/normal/98.mat", 
                                  "../data/normal/99.mat", 
                                  "../data/normal/100.mat",
                                 ]),
    
    "Inner Race": load_class_signals(["../data/12k drive end/InnerRace/105.mat", 
                                 "../data/12k drive end/InnerRace/106.mat", 
                                 "../data/12k drive end/InnerRace/107.mat", 
                                 "../data/12k drive end/InnerRace/108.mat",
                                ]),
    
    "Ball": load_class_signals(["../data/12k drive end/Ball/118.mat", 
                           "../data/12k drive end/Ball/119.mat",
                           "../data/12k drive end/Ball/120.mat", 
                           "../data/12k drive end/Ball/121.mat",
                          ]),
    
    "Outer Race_Center": load_class_signals(["../data/12k drive end/OuterRace_Center/130.mat", 
                                        "../data/12k drive end/OuterRace_Center/131.mat",
                                        "../data/12k drive end/OuterRace_Center/132.mat", 
                                        "../data/12k drive end/OuterRace_Center/133.mat",
                                       ]),
    
    "Outer Race_Ortho": load_class_signals(["../data/12k drive end/OuterRace_Ortho/156.mat", 
                                       "../data/12k drive end/OuterRace_Ortho/158.mat",
                                       "../data/12k drive end/OuterRace_Ortho/159.mat", 
                                       "../data/12k drive end/OuterRace_Ortho/160.mat",
                                      ]),
    
    "Outer Race_Oppo": load_class_signals(["../data/12k drive end/OuterRace_Oppo/144.mat", 
                                      "../data/12k drive end/OuterRace_Oppo/145.mat",
                                      "../data/12k drive end/OuterRace_Oppo/146.mat", 
                                      "../data/12k drive end/OuterRace_Oppo/147.mat",
                                     ]),
}
for name, sig in signals.items():
    print(f"{name:22s}: {len(sig):>7} samples = {len(sig)/FS:.1f}s")

fault_length = min(
    len(signals["Inner Race"]),
    len(signals["Ball"]),
    len(signals["Outer Race_Center"]),
    len(signals["Outer Race_Oppo"]),
    len(signals["Outer Race_Ortho"]),
)
signals["Normal"] = signals["Normal"][:fault_length]

#Sanity check
for name, sig in signals.items():
    n_windows = len(range(0, len(sig) - WINDOW_SIZE, STEP_SIZE))
    print(f"{name:22s}: {len(sig):>7} samples = {len(sig)/FS:.1f}s -> {n_windows} windows")

Normal                : 1697387 samples = 141.4s
Inner Race            :  488309 samples = 40.7s
Ball                  :  487093 samples = 40.6s
Outer Race_Center     :  488398 samples = 40.7s
Outer Race_Ortho      :  488689 samples = 40.7s
Outer Race_Oppo       :  487964 samples = 40.7s
Normal                :  487093 samples = 40.6s -> 80 windows
Inner Race            :  488309 samples = 40.7s -> 80 windows
Ball                  :  487093 samples = 40.6s -> 80 windows
Outer Race_Center     :  488398 samples = 40.7s -> 80 windows
Outer Race_Ortho      :  488689 samples = 40.7s -> 80 windows
Outer Race_Oppo       :  487964 samples = 40.7s -> 80 windows


In [24]:
def extract_features(sig: np.ndarray) -> dict:
    """Extract time-domain features from a single signal window"""
    rms = np.sqrt(np.mean(sig**2))
    kurtosis = stats.kurtosis(sig)
    crest_factor = np.max(np.abs(sig)) / rms
    peak_to_peak = np.max(sig) - np.min(sig)
    skewness = stats.skew(sig)
    return {
        "rms": rms,
        "kurtosis": kurtosis,
        "crest_factor": crest_factor,
        "peak_to_peak": peak_to_peak,
        "skewness": skewness,
    }

def build_feature_matrix(signals: dict, window_size: int, step_size: int) -> pd.DataFrame:
    """Slide a window over each Signal and extract features per window."""
    rows = []
    for label, sig in signals.items():
        starts = range(0, len(sig) - window_size, step_size)
        for start in starts:
            window = sig[start: start + window_size]
            features = extract_features(window)
            features["label"] = label
            rows.append(features)
    return pd.DataFrame(rows)

df_features = build_feature_matrix(signals, WINDOW_SIZE, STEP_SIZE)

print(f"Total windows: {len(df_features)}")
print(f"Per class:\n{df_features['label'].value_counts()}")

Total windows: 480
Per class:
label
Normal               80
Inner Race           80
Ball                 80
Outer Race_Center    80
Outer Race_Ortho     80
Outer Race_Oppo      80
Name: count, dtype: int64


In [25]:
def compute_spectrogram_patch(sig: np.ndarray, fs:int):
    """
    Compute a single STFT spectogram patch from a signal window.
    Returns a 2D array (frequency bins x time frames) in dB scale.
    """

    freqs, times, Zxx = signal.stft(
    sig, fs=fs,
    window = "hann",
    nperseg = 256,
    noverlap =192,
    boundary = None
    )

    freq_mask = freqs <= 3000
    power_db = 20 * np.log10(np.abs(Zxx[freq_mask]) + 1e-10)
    return power_db

def build_spectrogram_dataset(signals: dict, window_size: int, step_size:int, fs: int):
    """
    Build dataset of spectogram patches and corresponding labels.
    Returns X (patches) and y (integer labels).
    """
    label_map = {"Normal": 0, "Inner Race": 1, "Ball": 2, "Outer Race_Center": 3, "Outer Race_Oppo": 4, "Outer Race_Ortho": 5}

    X, y = [], []

    for label, sig in signals.items():
        starts = range(0, len(sig) - window_size, step_size)
        for start in starts:
            window = sig[start : start + window_size]
            patch = compute_spectrogram_patch(window, fs)
            X.append(patch)
            y.append(label_map[label])

    X = np.array(X) # shape: (n_windows, freq_bins, time_frames)
    y = np.array(y)
    return X, y, label_map

X, y, label_map = build_spectrogram_dataset(signals, WINDOW_SIZE, STEP_SIZE, FS)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Label map: {label_map}")

X shape: (480, 65, 185)
y shape: (480,)
Label map: {'Normal': 0, 'Inner Race': 1, 'Ball': 2, 'Outer Race_Center': 3, 'Outer Race_Oppo': 4, 'Outer Race_Ortho': 5}


In [26]:
# Saving feature matrix and spectrogram patches
np.save("../models/X_patches.npy", X)
np.save("../models/y_labels.npy", y)
df_features.to_csv("../models/features.csv", index=False)

print("Saved:")
print(f"X_patches.npy -> {X.shape}")
print(f"y_labels.npy -> {y.shape}")
print(f"features.csv -> {df_features.shape}")

Saved:
X_patches.npy -> (480, 65, 185)
y_labels.npy -> (480,)
features.csv -> (480, 6)
